# Minute-level price direction on NSE trade and quote data

One trading day of tick-level trade and quote records for 31 NIFTY names, aggregated to
minute bars and used to ask a single question: **can the next minute's direction be
predicted well enough to trade?**

The short answer, established below, is **no** — and the interesting part is *how* the
earlier version of this notebook concluded otherwise.

Every stage here is a call into `src/nse_pipeline/`, which is unit-tested and run in CI.
This notebook is the narrative; it is not where the logic lives.

In [1]:
import pandas as pd

from nse_pipeline import aggregate, backtest, config, db, features, ingest, model, split

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

## 1. Ingest

The raw feed is one headerless `.asc` file per symbol per side of the book. Paths come
from the environment (`NSE_TRADES_DIR` / `NSE_QUOTES_DIR`), so this runs the same way on a
laptop, in Docker and in CI.

In [2]:
trades = ingest.add_minute(ingest.load_trades(config.trades_dir()))
quotes = ingest.add_minute(ingest.load_quotes(config.quotes_dir()))

print(f"{len(trades):>10,} trade ticks")
print(f"{len(quotes):>10,} quote ticks")
print(f"{trades['symbol'].nunique():>10} symbols")

 2,933,072 trade ticks
 6,449,315 quote ticks
        31 symbols


## 2. Aggregate to minute bars

Nine million ticks become a few thousand bars: open/high/low/close, traded volume, VWAP,
and the spread statistics from the quote side. Trades and quotes are joined on the
composite `(symbol, minute)` key.

The join is an inner join deliberately — a minute with trades but no quotes has no spread,
and a half-populated row would quietly become a feature vector full of nulls.

In [3]:
bars = aggregate.merge_bars(aggregate.minute_bars(trades), aggregate.quote_stats(quotes))
print(f"{len(bars):,} minute bars")
bars.head()

11,668 minute bars


,symbol,minute,o,h,l,c,total_volume,num_trades,weighted_price,avg_spread,max_spread,min_spread,total_bid_size,total_ask_size,weighted_avg_bid_price,weighted_avg_ask_price
0,ADANIENT,2024-04-01 09:07:00,3230.20,3230.20,3230.2,3230.20,6720,386,3230.200000,-81.331402,1.00,-638.60,76268,128052,3258.433120,3214.149230
1,ADANIENT,2024-04-01 09:15:00,3230.20,3244.15,3215.0,3240.75,29934,1334,3233.390402,3.222845,15.95,0.05,908909,385346,3233.507170,3234.425155
2,ADANIENT,2024-04-01 09:16:00,3242.15,3244.90,3232.4,3242.70,20370,940,3238.166340,1.985751,5.25,0.05,302353,238905,3237.984834,3238.809164
3,ADANIENT,2024-04-01 09:17:00,3244.85,3248.10,3230.0,3232.00,16337,1063,3238.894822,1.735203,3.95,0.05,232745,159182,3235.273606,3238.981134
4,ADANIENT,2024-04-01 09:18:00,3232.35,3235.65,3226.0,3233.45,13953,742,3231.073339,1.630432,3.65,0.05,142395,88973,3230.081325,3232.145995


## 3. Load the warehouse

The bars go into SQLite once. Every question below is then a query rather than a re-run of
the pipeline, and the queries live in `sql/` as files that CI executes.

In [4]:
conn = db.connect(config.database_path())
print(f"{db.load_minute_bars(conn, bars):,} rows written to {config.database_path().name}")

11,668 rows written to nse.db


### The pre-open auction

The first query run against this warehouse returned a **negative** mean spread for the
09:00 hour. That is not bad data: NSE runs a pre-open call auction from 09:00 to 09:15, and
the book is crossed by design during order collection while a single opening price is
discovered.

Averaging those minutes in with the continuous session quietly poisons every spread
statistic, so they are reported separately.

In [5]:
db.query(conn, "crossed_book_minutes.sql")

,session,bars,crossed_bars,crossed_pct,mean_spread
0,continuous session,11637,0,0.0,0.8647
1,pre-open auction,31,31,100.0,-77.5158


In [6]:
db.query(conn, "liquidity_by_symbol.sql").head(10)

,symbol,bars,traded_volume,trades,mean_spread,mean_spread_bps,mean_resting_size
0,HDFCBANK,376,12596064,265982,0.0856,0.58,1718784.7
1,ICICIBANK,377,10846851,191633,0.0827,0.75,1989651.4
2,ITC,376,9380938,118241,0.0732,1.71,2653342.1
3,COALINDIA,376,7255701,82411,0.0990,2.24,1621519.9
4,INFY,377,6578412,233313,0.1043,0.69,1424121.5
5,HINDALCO,377,6192020,67130,0.1148,2.02,609840.2
6,JSWSTEEL,376,6053016,154764,0.2538,2.92,716232.0
7,AXISBANK,377,5484130,181128,0.1177,1.11,641833.9
8,BHARTIARTL,376,5461439,215291,0.1265,1.04,675181.8
9,HDFCLIFE,377,3873529,55658,0.1100,1.73,415065.5


### The base rate

This is the number every result below has to be compared against: how often does the next
minute close higher anyway? Computed in SQL with a window function, so it does not depend
on the Python feature code being correct.

In [7]:
base = db.query(conn, "direction_base_rate.sql")
print(f"overall up-rate: {base['up_bars'].sum() / base['labelled_bars'].sum():.4f}")
base.head()

overall up-rate: 0.4652


,symbol,labelled_bars,up_bars,up_rate_pct
0,TCS,375,191,50.93
1,ADANIPORTS,376,190,50.53
2,DRREDDY,375,188,50.13
3,DIVISLAB,375,187,49.87
4,HDFCBANK,375,185,49.33


## 4. Features and a leakage-safe split

Rolling means and standard deviations per symbol, then the next-minute direction label.

Two guards matter here:

- `feature_columns()` excludes `future_price`, `future_return` and `target` **by
  construction**, so leakage is structural rather than something to remember.
- The split is the tail of each symbol's own day. A random split would let the model train
  on minutes that come after the ones it is scored on.

In [8]:
frame = features.add_direction_target(features.add_rolling_features(bars))
train_df, test_df = split.time_ordered_split(frame, holdout=100)
feature_cols = features.feature_columns(frame)

print(f"{len(feature_cols)} features")
print(f"train {len(train_df):,}   test {len(test_df):,}")
print(f"leaked columns in the feature set: "
      f"{[c for c in feature_cols if 'future' in c or c == 'target']}")

38 features
train 8,537   test 3,100
leaked columns in the feature set: []


## 5. The model

A random forest, scored on **precision at a confidence cutoff** rather than accuracy. A
model that is right 51% of the time on every minute is useless; one that is right 60% of the
time on the few minutes it is confident about could be traded.

The cutoff sweep is the honest way to look at this, because it shows the price of
precision: every step up in confidence buys accuracy with trade count.

In [9]:
forest = model.train(train_df[feature_cols], train_df["target"])
probabilities = forest.predict_proba(test_df[feature_cols])[:, 1]

sweep = model.threshold_sweep(test_df["target"], probabilities)
sweep.round(4)

,threshold,trades,precision,recall,base_rate,edge_over_base
0,0.50,1026,0.4571,0.3230,0.4684,-0.0113
1,0.55,443,0.4357,0.1329,0.4684,-0.0327
2,0.60,146,0.4452,0.0448,0.4684,-0.0232
3,0.63,61,0.4590,0.0193,0.4684,-0.0094
4,0.65,32,0.5000,0.0110,0.4684,0.0316
5,0.70,10,0.3000,0.0021,0.4684,-0.1684


**Read the `edge_over_base` column, not the `precision` column.**

Precision sits *below* the base rate at every cutoff but one, and that one fires on 32
trades. An earlier version of this notebook reported precision 0.6143 at threshold 0.63 and
stopped there — but that cutoff was chosen by sweeping thresholds on the test set and
reporting the best one. On 70 trades, that is noise.

The ROC-AUC settles it.

In [10]:
summary = model.summarise(test_df["target"], probabilities, threshold=0.63)
print(f"ROC-AUC   {summary['roc_auc']:.4f}")
print(f"base rate {summary['base_rate']:.4f}")
print(f"precision {summary['precision']:.4f}  on {summary['trades']} trades")
print(f"edge      {summary['edge_over_base']:+.4f}")

ROC-AUC   0.4937
base rate 0.4684
precision 0.4590  on 61 trades
edge      -0.0094


ROC-AUC below 0.5 means the ranking carries no information at all. Buying at random
would have done as well.

## 6. The backtest, with costs

The original backtest bought a fixed amount whenever the model was confident, summed the
outcome, and reported a profit. What it did not report was the cost of the trades that
produced it — which is the only thing that decides whether a strategy is real.

`buy_and_hold` is here as the benchmark: a strategy that makes money is uninteresting if
doing nothing made more.

In [11]:
signals = (probabilities >= 0.63).astype(int)

for label, bps in [("no costs (as originally run)", 0), ("10bp round trip", 10)]:
    result = backtest.run(test_df, signals, stake=200.0, cost_bps=bps)
    print(f"{label:30} trades={result['trades']:4d}  "
          f"gross={result['gross_pnl']:+8.2f}  costs={result['costs']:6.2f}  "
          f"net={result['net_pnl']:+8.2f}  survives={result['survives_costs']}")

print(f"{'buy and hold':30} {backtest.buy_and_hold(test_df)}")

no costs (as originally run)   trades=  61  gross=   -0.17  costs=  0.00  net=   -0.17  survives=False
10bp round trip                trades=  61  gross=   -0.17  costs= 12.20  net=  -12.37  survives=False
buy and hold                   {'bars': 3100, 'pnl': 4.806481235461786}


## What this project actually shows

1. **A pipeline that holds up.** Nine million ticks to a queryable warehouse, with the
   aggregation, the features and the split all unit-tested and run in CI.
2. **A real microstructure finding.** The pre-open auction crosses the book, and mixing
   those minutes into session statistics is a mistake that is easy to make and invisible
   once made.
3. **A negative result, honestly reported.** There is no tradeable next-minute edge in
   this feature set on this day. The earlier 0.6143 was a threshold picked on the test set,
   and the backtest that "profited" lost money once its own costs were charged.

The third point is the most useful one. A backtest that lies is worse than no backtest, and
this repository is named after that failure mode.

### What would make the question worth re-asking

- More than one trading day, so a result can be checked out of sample rather than
  in-sample at a hand-picked cutoff.
- Order-book depth beyond the top level, which is where short-horizon signal usually lives.
- Choosing the cutoff on a validation split, never on the test set.